# EEG NCDF Batch Quality Report

This notebook crawls the exported EEG tree, runs AutoReject quality checks for each EEG NCDF file, and saves per-file CSV + PNG artifacts in the same format as the single-file demo.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

sys.path.insert(0, os.path.abspath('..'))

from src.export import run_eeg_autoreject_quality_report

In [ ]:
# Prefer setting the UNIWAW_EEG_EXPORT_FOLDER environment variable; otherwise, update the placeholder below.
export_folder_env = os.getenv("UNIWAW_EEG_EXPORT_FOLDER")
export_folder = Path(export_folder_env) if export_folder_env else Path("/path/to/UNIWAW_EEG_exported")

if not export_folder_env and export_folder == Path("/path/to/UNIWAW_EEG_exported"):
    raise ValueError(
        "export_folder is not configured. "
        "Set the UNIWAW_EEG_EXPORT_FOLDER environment variable or update the placeholder path above."
    )

# Crawl full tree and collect only EEG NCDF files.
eeg_files = sorted([p for p in export_folder.rglob("*.nc") if "_EEG_" in p.name])

if not eeg_files:
    raise FileNotFoundError(f"No EEG NetCDF files found under: {export_folder}")

print(f"Found {len(eeg_files)} EEG files.")

In [ ]:
# Smoke-test switch: process only the first N files before full batch.
smoke_test = False
subset_size = 12

files_to_process = eeg_files[:subset_size] if smoke_test else eeg_files
mode = f"SMOKE TEST (first {subset_size})" if smoke_test else "FULL BATCH"
print(f"Mode: {mode}")
print(f"Will process {len(files_to_process)} / {len(eeg_files)} files")
for p in files_to_process:
    print(f"- {p.name}")

In [ ]:
def _toml_scalar(value):
    if isinstance(value, bool):
        return "true" if value else "false"
    if isinstance(value, (int, float)):
        return str(value)
    if value is None:
        return '""'
    text = str(value).replace('"', '\\"')
    return f'"{text}"'


def _extract_dyad_name(ncdf_path: Path):
    parts = ncdf_path.stem.split("_")
    if len(parts) >= 2:
        return f"{parts[0]}_{parts[1]}"
    return ncdf_path.stem


def _get_top_bad_channels(report, threshold=10):
    channel_summary = report["channel_summary"].copy()
    bad_channels = channel_summary.loc[channel_summary["bad_labels_pct"] > threshold, "channel"].tolist()
    return ", ".join(bad_channels) if bad_channels else ""


def _build_report_rows(report, ncdf_path: Path):
    channel_summary = report["channel_summary"].copy()
    epoch_summary = report["epoch_summary"].copy()
    global_summary = dict(report["global_summary"])

    global_summary["ncdf_file"] = ncdf_path.name
    global_summary.pop("ncdf_path", None)
    global_summary.pop("n_channels", None)
    global_summary.pop("n_epochs", None)

    rows = []

    # 1. File name first.
    rows.append({"section": "global", "key": "ncdf_file", "value": _toml_scalar(global_summary["ncdf_file"])})

    # 2. Rejected windows excluding margin epochs.
    rejected_epochs = epoch_summary.loc[epoch_summary["rejected"] & ~epoch_summary["in_margin"]]
    rows.append({
        "section": "global",
        "key": "rejected_epochs",
        "value": str(int(len(rejected_epochs))),
    })
    for _, row in rejected_epochs.iterrows():
        rows.append({
            "section": "rejected_windows",
            "key": f"epoch_{int(row['epoch_idx'])}",
            "value": (
                f"{{start_s={float(row['start_s']):.3f}, "
                f"end_s={float(row['end_s']):.3f}, "
                f"interpolated_channels={int(row['interpolated_channels'])}}}"
            ),
        })

    # 3. Remaining global values.
    skip_keys = {"ncdf_file", "n_channels", "n_epochs"}
    for key, value in global_summary.items():
        if key not in skip_keys:
            rows.append({"section": "global", "key": key, "value": _toml_scalar(value)})

    # 4. Top channels with percentages only (rounded).
    top_channels = channel_summary.head(10)
    for idx, row in top_channels.iterrows():
        problematic_pct = int(round(float(row["interpolated_pct"]) + float(row["bad_labels_pct"])))
        fixable_pct = int(round(float(row["interpolated_pct"])))
        still_bad_pct = int(round(float(row["bad_labels_pct"])))
        rows.append({
            "section": "top_channels",
            "key": f"ch_{idx + 1}",
            "value": (
                f"{{name={_toml_scalar(row['channel'])}, "
                f"problematic_pct={problematic_pct}, "
                f"fixable_pct={fixable_pct}, "
                f"still_bad_pct={still_bad_pct}}}"
            ),
        })

    return pd.DataFrame(rows)


def _save_report_artifacts(report, ncdf_path: Path):
    out_dir = ncdf_path.parent
    base_name = ncdf_path.stem
    report_csv = out_dir / f"{base_name}_quality_report.csv"
    plot_png = out_dir / f"{base_name}_quality_plot.png"

    rows_df = _build_report_rows(report, ncdf_path)
    rows_df.to_csv(report_csv, index=False)
    report["figure"].savefig(plot_png, dpi=200, bbox_inches="tight")
    return report_csv, plot_png, rows_df

In [ ]:
results = []

for idx, ncdf_path in enumerate(files_to_process, start=1):
    dyad_name = _extract_dyad_name(ncdf_path)
    display(Markdown(f"## Dyad: **{dyad_name}**"))
    print(f"\n{'='*80}")
    print(f"[{idx}/{len(files_to_process)}] Processing: {ncdf_path.name}")

    try:
        report = run_eeg_autoreject_quality_report(
            ncdf_path=str(ncdf_path),
            epoch_duration_s=2.0,
            n_interpolate=(1, 2, 4),
            cv=5,
            random_state=42,
            n_jobs=-1,
            montage="standard_1020",
            scale_to_volts=1e-6,
            verbose=False,
        )

        report_csv, plot_png, rows_df = _save_report_artifacts(report, ncdf_path)
        top_bad_channels = _get_top_bad_channels(report, threshold=10)

        print(f"Saved CSV: {report_csv}")
        print(f"Saved PNG: {plot_png}")
        print("Summary written to CSV (global + rejected windows + top channels):")
        display(rows_df)

        display(report["figure"])
        plt.close(report["figure"])

        rejected_row = rows_df[(rows_df["section"] == "global") & (rows_df["key"] == "rejected_epochs")]
        rejected_epochs = int(rejected_row.iloc[0]["value"]) if not rejected_row.empty else None

        results.append({
            "ncdf_file": ncdf_path.name,
            "dyad": dyad_name,
            "status": "ok",
            "rejected_epochs": rejected_epochs,
            "top_bad_channels": top_bad_channels,
        })
    except Exception as exc:
        print(f"FAILED: {ncdf_path.name} -> {exc}")
        results.append({
            "ncdf_file": ncdf_path.name,
            "dyad": dyad_name,
            "status": "failed",
            "rejected_epochs": None,
            "top_bad_channels": "",
            "error": str(exc),
        })

In [ ]:
results_df = pd.DataFrame(results)
summary_columns = ["dyad", "ncdf_file", "status", "rejected_epochs", "top_bad_channels"]
if "error" in results_df.columns:
    summary_columns.append("error")
summary_report_df = results_df.reindex(columns=summary_columns)

summary_report_path = export_folder / "EEG_quality_summary_report.csv"
summary_report_df.to_csv(summary_report_path, index=False)

print("\nBatch run summary:")
display(summary_report_df)
print(f"Saved summary report: {summary_report_path}")

n_ok = int((results_df["status"] == "ok").sum()) if not results_df.empty else 0
n_fail = int((results_df["status"] == "failed").sum()) if not results_df.empty else 0
print(f"Completed. OK: {n_ok}, Failed: {n_fail}")